In [2]:
%pip install -q transformers torch pandas matplotlib seaborn requests tqdm ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [10]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import torch
from IPython.display import Markdown, display
from ipywidgets import Dropdown, interact
from tqdm.auto import tqdm
from transformers import pipeline

pd.set_option("display.max_columns", 160)
sns.set_theme(style = "whitegrid")
print("PyTorch:", torch.__version__)
print("CUDA-GPU tilgængelig:", torch.cuda.is_available())

PyTorch: 2.14.0+cpu
CUDA-GPU tilgængelig: False


In [14]:
TEXT_URL = "https://www.gutenberg.org/cache/epub/97/pg97.txt"
TEXT_FILE = Path("flatland.txt")
if not TEXT_FILE.exists():
    response = requests.get(TEXT_URL, timeout=30)
    response.raise_for_status()
    TEXT_FILE.write_text(response.text, encoding="utf-8")
raw_text = TEXT_FILE.read_text(encoding="utf-8")
print(f"Indlæst {len(raw_text):,} tegn")
print(raw_text[:300])

Indlæst 213,450 tegn
The Project Gutenberg eBook of Flatland: A Romance of Many Dimensions
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Proj


In [17]:
def extract_book(text):
    start = text.find("§ 1 Of the Nature of Flatland")
    end = text.find("*** END OF THE PROJECT GUTENBERG EBOOK")
    if start == -1:
        raise ValueError("Kunne ikke finde første afsnit")
    return text[start:end if end != -1 else None]


def parse_sections(text):
    heading = re.compile(r"^§\s*(\d+)\s+(.+)$")
    records = []
    number = None
    title = None
    buffer = []

    def save():
        nonlocal buffer, number, title
        if number is not None and title is not None:
            body = "\n".join(buffer)
            body = re.sub(r"\[Illustration.*?\]", " ", body, flags=re.I | re.S)
            body = re.sub(r"[ \t]+", " ", body)
            body = re.sub(r"\n{3,}", "\n\n", body).strip()
            records.append(
                {
                    "section": number,
                    "title": title.strip(),
                    "part": "I — This World" if number <= 12 else "II — Other Worlds",
                    "text": body,
                }
            )
        buffer = []

    lines = text.splitlines()
    i = 0

    while i < len(lines):
        line = lines[i].strip()
        match = heading.match(line)

        if match:
            save()
            number = int(match.group(1))
            title = match.group(2)

            # Nogle lange overskrifter fortsætter på næste linje.
            while i + 1 < len(lines):
                nxt = lines[i + 1].strip()
                if not nxt or heading.match(nxt):
                    break
                title += " " + nxt
                i += 1

        elif number is not None:
            buffer.append(line)

        i += 1

    save()

    df = pd.DataFrame(records)
    df["word_count"] = df["text"].str.split().apply(len)
    return df


book_text = extract_book(raw_text)
sections = parse_sections(book_text)
print(f"Fandt {len(sections)} afsnit og {sections.word_count.sum():,} ord")
sections[["section", "part", "title", "word_count"]]

Fandt 22 afsnit og 33,244 ord


,section,part,title,word_count
0,1,I — This World,Of the Nature of Flatland,731
1,2,I — This World,Of the Climate and Houses in Flatland,807
2,3,I — This World,Concerning the Inhabitants of Flatland,1218
3,4,I — This World,Concerning the Women,1899
4,5,I — This World,Of our Methods of Recognizing one another,1763
5,6,I — This World,Of Recognition by Sight,1742
6,7,I — This World,Concerning Irregular Figures,1216
7,8,I — This World,Of the Ancient Practice of Painting,974
8,9,I — This World,Of the Universal Colour Bill,1293
9,10,I — This World,Of the Suppression of the Chromatic Sedition,1449
